# RZ-USFM project · Part 5 — External validation on BUSI (no retraining on BUSI)

Every model is **developed on BUS-BRA only** and applied **unchanged** to BUSI (a different country, scanner and annotation protocol).

**Protocol**
1. **BUSI curation:** benign + malignant only, multiple masks merged, duplicates / near-duplicates removed with perceptual hashing (conflicting labels → both removed).
2. **Experts on BUSI = average of their 5 cross-validation models** (the models of Parts 1–3), the same rule for every expert:
   - **A** (radiomics): the per-fold pipelines of Part 1 are refitted (same features, same classifier) and applied to BUSI;
   - **B** (deep dual-stream): Part 2 saved weights only for one model, so the missing fold × seed models are **re-trained with the identical code, splits and seeds** (this time keeping the weights); their BUS-BRA test-fold predictions replace Part 2's for full consistency;
   - **C** (RZ-USFM) and **USFM-FT**: all saved LoRA models of Part 3.
3. **Radiomic concepts and consistency score $c$ on BUSI** with the fold-wise transforms of Part 1 and residual covariances of Part 3 (all fitted on BUS-BRA).
4. **Fusion models refitted on all BUS-BRA out-of-fold predictions**, applied to BUSI. BUSI is an **unseen scanner**: the hierarchical model uses $\alpha_{k,\text{new}}=\bar\alpha_k+\tau_k z$, $z\sim\mathcal N(0,1)$.
5. **Thresholds and conformal cut-offs are fixed on BUS-BRA** before BUSI is touched.

**Questions answered:** which expert degrades least · RZ-USFM vs USFM-FT under shift · does $c$ rise and flag errors on BUSI · gated vs ungated fusion under shift · does the conformal sensitivity guarantee transfer · triage with bootstrap CIs and McNemar tests (BUS-BRA BI-RADS 4 and BUSI).

## 1. Setup and paths

In [ ]:
!pip -q install "git+https://github.com/AIM-Harvard/pyradiomics" SimpleITK timm tabpfn pymc arviz gdown
!git clone -q https://github.com/openmedlab/USFM.git /content/USFM 2>/dev/null || echo 'USFM repo already present'
from google.colab import drive
drive.mount('/content/drive')
import os, sys, glob, re, json, time, zipfile, shutil, warnings
warnings.filterwarnings('ignore')

D = '/content/drive/MyDrive'
PART1_DIR = f'{D}/rsusfm_project'
PART5_DIR = f'{D}/rsusfm_project_part5'                       # ALL outputs of this notebook
BUSI_DIR, BUSI_ZIP = f'{D}/BUSI', f'{D}/BUSI.zip'              # either the folder or the zip
USFM_REPO, USFM_CKPT = '/content/USFM', f'{D}/weights/USFM_latest.pth'

def first(cands, pattern):
    d = next((c for c in cands if glob.glob(f'{c}/{pattern}')), None)
    assert d, f'{pattern} not found in {cands}'; return d
P1_RES = f'{PART1_DIR}/results'
P2_RES = first([f'{D}/rsusfm_project_part2/results', P1_RES], 'oof_expertB.csv')
P2_RUN = first([f'{D}/rsusfm_project_part2/runs', f'{P1_RES}/expertB_runs'], 'dual_f*_s*.npz')
P3_RES = first([f'{D}/rsusfm_project_part3/results', P1_RES], 'oof_expertC.csv')
P3_RUN = first([f'{D}/rsusfm_project_part3/runs', f'{P1_RES}/expertC_runs'], 'lam*_f*_s*.pt')
P4_RES = next((c for c in [f'{D}/rsusfm_project_part4/results', P1_RES] if os.path.exists(f'{c}/fusion_oof_cache.npz')), None)
RES_DIR, FIG_DIR = f'{PART5_DIR}/results', f'{PART5_DIR}/figures'
B_RUN, RAD5 = f'{PART5_DIR}/expertB_models', f'{PART5_DIR}/radiomics'
for d in (RES_DIR, FIG_DIR, B_RUN, RAD5, f'{PART5_DIR}/busi_masks'): os.makedirs(d, exist_ok=True)
for k, v in dict(P1_RES=P1_RES, P2_RES=P2_RES, P2_RUN=P2_RUN, P3_RES=P3_RES, P3_RUN=P3_RUN, P4_RES=P4_RES).items(): print(f'{k:7s}', v)

# ---- must match Parts 2 and 3 ----
BACKBONE, PRETRAINED = 'convnext_tiny.fb_in22k_ft_in1k', True
IMG, CACHE_SIZE = 224, 288
EPOCHS_B, BS_B, LR_BACKBONE, LR_HEAD, WD_B = 20, 32, 1e-4, 1e-3, 0.05
N_SEEDS_B, N_SEEDS_C = 3, 3
LORA_RANK, LORA_ALPHA = 8, 16
ICC_MIN, N_BOOT_TABPFN = 0.75, 5
SEED = 42
ALPHA_MISS, ALPHA_FP = 0.02, 0.10
DRAWS, TUNE, CHAINS = 1000, 1000, 2
HASH_THR, CORR_THR = 4, 0.95       # duplicate = perceptual-hash distance <= 4 bits AND pixel correlation >= 0.95
# FAST OPTION for Expert B (4 models instead of 14, about 30 min): N_SEEDS_B = 1

try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN'); print('HF_TOKEN loaded (TabPFN)')
except Exception as e:
    print('No HF_TOKEN secret -> TabPFN unavailable; Expert A uses the classical model of Part 1 if that was used')

In [ ]:
sys.path.insert(0, PART1_DIR); sys.path.insert(0, PART5_DIR)
import importlib, common; importlib.reload(common)
from common import *
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F, timm
from scipy import stats, ndimage as ndi
from PIL import Image
from joblib import Parallel, delayed
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)
master = pd.read_csv(f'{PART1_DIR}/master.csv')
y_bb, g_bb = master.label.values, master.patient.astype(str).values
FOLDS = sorted(master.fold.unique())
dev = master['device'].astype(str).values if 'device' in master else np.array(['all'] * len(master))
DEV_LEVELS = sorted(np.unique(dev)); S_IDX = np.array([DEV_LEVELS.index(d) for d in dev])
birads = master['birads'].astype(str).str.strip().str.upper() if 'birads' in master else pd.Series(['NA'] * len(master))
BR4 = birads.str.startswith('4').values
print(f'BUS-BRA: {len(master)} lesions, {BR4.sum()} BI-RADS 4, scanners {DEV_LEVELS}')

## 2. BUSI curation: pairing, mask merging, duplicate removal
Labels come from the folder names (`benign`, `malignant`; `normal` is ignored). Images with several masks (`_mask_1.png`, …) get the **union** of their masks. Duplicates: perceptual hash (DCT, 63 bits) within `HASH_THR` bits **and** pixel correlation ≥ `CORR_THR` (64×64 thumbnails). Each duplicate group keeps one image, or is removed entirely if its labels disagree.

In [ ]:
from scipy.fft import dctn
root = BUSI_DIR
if not os.path.isdir(BUSI_DIR) and os.path.exists(BUSI_ZIP):
    root = '/content/BUSI_raw'
    if not os.path.isdir(root):
        with zipfile.ZipFile(BUSI_ZIP) as z: z.extractall(root)
assert os.path.isdir(root), 'BUSI folder / zip not found - check BUSI_DIR / BUSI_ZIP'

rows = []
for d in sorted(glob.glob(f'{root}/**/', recursive=True)):
    lab = os.path.basename(os.path.normpath(d)).lower()
    if lab not in ('benign', 'malignant'): continue
    files = sorted(os.listdir(d))
    for fn in files:
        if not fn.lower().endswith('.png') or '_mask' in fn.lower(): continue
        stem = fn[:-4]
        mks = [m for m in files if re.fullmatch(re.escape(stem) + r'_mask(_\d+)?\.png', m, flags=re.I)]
        num = re.search(r'\((\d+)\)', stem)
        rows.append({'id': f'busi_{lab}_{int(num.group(1)):03d}' if num else f'busi_{lab}_{re.sub(r"[^A-Za-z0-9]+", "_", stem)}',
                     'label': int(lab == 'malignant'), 'img_path': os.path.join(d, fn), 'masks': [os.path.join(d, m) for m in mks]})
busi = pd.DataFrame(rows)
print(f'found {len(busi)} images ({busi.label.sum()} malignant) | without mask: {(busi.masks.str.len() == 0).sum()} | '
      f'with several masks: {(busi.masks.str.len() > 1).sum()}')
busi = busi[busi.masks.str.len() > 0].reset_index(drop=True)

def merge_masks(r):
    out = f'{PART5_DIR}/busi_masks/{r.id}.png'
    if not os.path.exists(out):
        shape = np.array(Image.open(r.img_path).convert('L')).shape
        m = np.zeros(shape, bool)
        for p in r.masks:
            a = np.array(Image.open(p).convert('L'))
            if a.shape != shape: a = np.array(Image.fromarray(a).resize(shape[::-1], Image.NEAREST))
            m |= a > 127
        Image.fromarray(m.astype(np.uint8) * 255).save(out)
    return out
busi['mask_path'] = [merge_masks(r) for r in busi.itertuples()]
busi['mask_px'] = [int((np.array(Image.open(p)) > 127).sum()) for p in busi.mask_path]
busi = busi[busi.mask_px > 20].reset_index(drop=True)

def phash_bits(p):
    a = np.asarray(Image.open(p).convert('L').resize((32, 32), Image.LANCZOS), float)
    d = dctn(a, norm='ortho')[:8, :8].ravel()[1:]
    return d > np.median(d)
H = np.stack([phash_bits(p) for p in busi.img_path])
ham = (H[:, None, :] != H[None, :, :]).sum(-1)
parent = list(range(len(busi)))
def find(i):
    while parent[i] != i: parent[i] = parent[parent[i]]; i = parent[i]
    return i
thumb = np.stack([np.asarray(Image.open(p).convert('L').resize((64, 64), Image.BILINEAR), float).ravel() for p in busi.img_path])
thumb = (thumb - thumb.mean(1, keepdims=True)) / (thumb.std(1, keepdims=True) + 1e-8)
for i, j in zip(*np.where(np.triu(ham <= HASH_THR, 1))):           # candidate by hash, confirmed by pixel correlation
    if (thumb[i] * thumb[j]).mean() >= CORR_THR: parent[find(i)] = find(j)
busi['group'] = [find(i) for i in range(len(busi))]
keep, dup_pairs, n_conflict = [], [], 0
for gid, g in busi.groupby('group'):
    if len(g) == 1: keep.append(g.index[0]); continue
    dup_pairs.append(list(g.index[:2]))
    if g.label.nunique() > 1: n_conflict += len(g); continue
    keep.append(g.index[0])
n0 = len(busi); busi_all = busi.copy()
busi = busi.loc[sorted(keep)].reset_index(drop=True)
busi['patient'] = busi.id; busi['fold'] = 0
busi[['id', 'label', 'img_path', 'mask_path', 'mask_px']].to_csv(f'{RES_DIR}/busi_master.csv', index=False)
print(f'duplicate groups: {len(dup_pairs)} | removed {n0 - len(busi)} images ({n_conflict} with conflicting labels)')
print(f'BUSI after curation: {len(busi)} lesions ({busi.label.sum()} malignant, {(busi.label == 0).sum()} benign)')

if dup_pairs:
    k = min(6, len(dup_pairs)); fig, axes = plt.subplots(2, k, figsize=(2.6 * k, 5.2)); axes = np.array(axes).reshape(2, k)
    for j, (a, b) in enumerate(dup_pairs[:k]):
        for r_, i in enumerate((a, b)):
            axes[r_, j].imshow(Image.open(busi_all.img_path[i]).convert('L'), cmap='gray')
            axes[r_, j].set_title(f'{busi_all.id[i]} ({["B", "M"][busi_all.label[i]]})', fontsize=7); axes[r_, j].axis('off')
    fig.suptitle(f'Duplicate pairs (hash ≤ {HASH_THR} bits, correlation ≥ {CORR_THR}) — check visually'); fig.tight_layout()
    fig.savefig(f'{FIG_DIR}/busi_duplicates.png', bbox_inches='tight'); plt.show()

## 3. Radiomics on BUSI (identical settings to Part 1) and the fold-wise radiomic concepts
The concept transforms (standardisation + family-wise PCA of each training fold) are **re-built from BUS-BRA** exactly as in Part 1 and checked against Part 1's saved concepts before being applied to BUSI.

In [ ]:
import logging, SimpleITK as sitk
from radiomics import featureextractor
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
logging.getLogger('radiomics').setLevel(logging.ERROR)
SETTINGS = dict(force2D=True, force2Ddimension=0, binCount=32, normalize=True, normalizeScale=100,
                removeOutliers=3, minimumROIDimensions=1, label=1)
TEXTURE = ['firstorder', 'glcm', 'glrlm', 'glszm', 'gldm', 'ngtdm']

def make_extractor(with_shape):
    ex = featureextractor.RadiomicsFeatureExtractor(**SETTINGS)
    ex.disableAllImageTypes(); ex.disableAllFeatures()
    ex.enableImageTypeByName('Original'); ex.enableImageTypeByName('Wavelet')
    for c in TEXTURE + (['shape2D'] if with_shape else []): ex.enableFeatureClassByName(c)
    return ex

def ring_from(lesion, frac=0.20, min_px=5):
    diam = 2 * np.sqrt(lesion.sum() / np.pi); w = max(min_px, int(round(frac * diam)))
    dist = ndi.distance_transform_edt(~lesion); return (dist > 0) & (dist <= w)

def load_pair(img_path, mask_path):
    img = np.array(Image.open(img_path).convert('L'), dtype=np.float32)
    msk = np.array(Image.open(mask_path).convert('L')) > 127
    if msk.shape != img.shape:
        msk = np.array(Image.fromarray(msk.astype(np.uint8) * 255).resize(img.shape[::-1], Image.NEAREST)) > 127
    return img, msk

def extract_one(row_id, img_path, mask_path):
    try:
        img, les = load_pair(img_path, mask_path); ring = ring_from(les)
        I = sitk.GetImageFromArray(img[None]); out = {'id': row_id}
        for region, m, shape in (('lesion', les, True), ('ring', ring, False)):
            f = make_extractor(shape).execute(I, sitk.GetImageFromArray(m.astype(np.uint8)[None]))
            out.update({f'{region}__{k}': float(v) for k, v in f.items() if not k.startswith('diagnostics')})
        return out
    except Exception as e:
        print('failed', row_id, type(e).__name__, e); return {'id': row_id}

def extract_all(frame, tag, chunk=200):
    parts = sorted(glob.glob(f'{RAD5}/{tag}_part*.csv')); done = set()
    for p in parts: done |= set(pd.read_csv(p, usecols=['id']).id)
    todo = frame[~frame.id.isin(done)]; print(f'{tag}: {len(done)} cached, {len(todo)} to extract')
    for s in range(0, len(todo), chunk):
        sub = todo.iloc[s:s + chunk]
        rows_ = Parallel(n_jobs=-1)(delayed(extract_one)(r.id, r.img_path, r.mask_path) for r in sub.itertuples())
        pd.DataFrame(rows_).to_csv(f'{RAD5}/{tag}_part{len(parts) + s // chunk:03d}.csv', index=False)
    return pd.concat([pd.read_csv(p) for p in sorted(glob.glob(f'{RAD5}/{tag}_part*.csv'))]).drop_duplicates('id')

# BUS-BRA radiomics + stable features exactly as in Part 1
rad_bb = pd.concat([pd.read_csv(p) for p in sorted(glob.glob(f'{PART1_DIR}/radiomics/rad_orig_part*.csv'))]).drop_duplicates('id')
rad_bb = master[['id']].merge(rad_bb, on='id', how='left')
feat_cols = [c for c in rad_bb.columns if c != 'id']
icc = pd.read_csv(f'{PART1_DIR}/radiomics/icc.csv', index_col=0).iloc[:, 0].reindex(feat_cols).fillna(0)
var_ok = rad_bb[feat_cols].std() > 1e-8
stable = [c for c in feat_cols if icc[c] >= ICC_MIN and var_ok[c] and rad_bb[c].notna().mean() > 0.99]
X_bb = rad_bb.set_index('id').loc[master.id, stable]; MED = X_bb.median(); X_bb = X_bb.fillna(MED)

rad_busi = extract_all(busi, 'busi').set_index('id').reindex(busi.id)
X_busi = rad_busi.reindex(columns=stable).fillna(MED)
print(f'{len(stable)} stable features | BUSI rows with failed extraction: {rad_busi[stable].isna().all(1).sum()}')

def fclass(c):
    region, name = c.split('__'); return region, name.split('_')[1]
families = {}
for c in stable: families.setdefault('{}_{}'.format(*fclass(c)), []).append(c)
families = {k: v for k, v in families.items() if len(v) >= 2}
CONCEPT_NAMES = [c for c in pd.read_csv(f'{PART1_DIR}/radiomics/concepts_fold0.csv', nrows=1).columns if c != 'id']
K = len(CONCEPT_NAMES)
CONC_BB, CONC_BUSI = {}, {}
for f in FOLDS:
    tr = (master.fold != f).values; out_bb, out_ext = {}, {}
    for name, cols in families.items():
        sc = StandardScaler().fit(X_bb.loc[:, cols].values[tr])
        Ztr = sc.transform(X_bb.loc[:, cols].values[tr]); pca = PCA(1, random_state=SEED).fit(Ztr)
        s_all = pca.transform(sc.transform(X_bb.loc[:, cols].values))[:, 0]; s_tr = s_all[tr]
        sign = np.sign(np.corrcoef(s_tr, Ztr.mean(1))[0, 1]) or 1.0
        s_ext = pca.transform(sc.transform(X_busi.loc[:, cols].values))[:, 0]
        out_bb[name] = sign * (s_all - s_tr.mean()) / (s_tr.std() + 1e-8)
        out_ext[name] = sign * (s_ext - s_tr.mean()) / (s_tr.std() + 1e-8)
    CONC_BB[f] = pd.DataFrame(out_bb)[CONCEPT_NAMES].values.astype(np.float32)
    CONC_BUSI[f] = pd.DataFrame(out_ext)[CONCEPT_NAMES].values.astype(np.float32)
    saved = pd.read_csv(f'{PART1_DIR}/radiomics/concepts_fold{f}.csv').set_index('id').loc[master.id, CONCEPT_NAMES].values
    err = np.abs(saved - CONC_BB[f]).max()
    print(f'fold {f}: rebuilt concepts match Part 1 (max |diff| = {err:.2e})'); assert err < 1e-3
print('mean |concept| on BUS-BRA vs BUSI (fold 0):', np.abs(CONC_BB[0]).mean().round(2), 'vs', np.abs(CONC_BUSI[0]).mean().round(2))

## 4. Expert A on BUSI — the 5 fold pipelines of Part 1 (same selected features, same classifier)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
A_oof = pd.read_csv(f'{P1_RES}/oof_expertA.csv').set_index('id').loc[master.id].reset_index()
cands = {n: pd.read_csv(f'{P1_RES}/oof_radiomics_{n}.csv').set_index('id').loc[master.id].prob.values
         for n in ('TabPFN', 'LogReg', 'RandomForest') if os.path.exists(f'{P1_RES}/oof_radiomics_{n}.csv')}
A_NAME = min(cands, key=lambda n: np.abs(cands[n] - A_oof.prob.values).max())
print('Expert A model identified as:', A_NAME)
def make_A():
    if A_NAME == 'TabPFN':
        from tabpfn import TabPFNClassifier; return TabPFNClassifier(device=device, random_state=SEED)
    if A_NAME == 'LogReg': return LogisticRegression(C=0.1, class_weight='balanced', max_iter=5000)
    return RandomForestClassifier(500, min_samples_leaf=3, class_weight='balanced_subsample', n_jobs=-1, random_state=SEED)

def fit_predict(make, Xtr, ytr, Xte, gtr=None, n_boot=1):        # identical to Part 1
    if n_boot == 1:
        p = make().fit(Xtr, ytr).predict_proba(Xte)[:, 1]; return p, logit(p), np.full(len(p), np.nan)
    rng = np.random.default_rng(SEED); ug = np.unique(gtr); L = []
    for _ in range(n_boot):
        idx = np.concatenate([np.where(gtr == g)[0] for g in rng.choice(ug, len(ug))])
        L.append(logit(make().fit(Xtr[idx], ytr[idx]).predict_proba(Xte)[:, 1]))
    L = np.stack(L); mu = L.mean(0); return 1 / (1 + np.exp(-mu)), mu, L.std(0)

sel_feats = pd.read_csv(f'{P1_RES}/expertA_selected_features.csv')
LA, SA = [], []
for f in FOLDS:
    tr = (master.fold != f).values
    sc = StandardScaler().fit(X_bb.values[tr]); idx = [stable.index(c) for c in sel_feats[f'fold{f}'].dropna()]
    p, mu, sd = fit_predict(make_A, sc.transform(X_bb.values[tr])[:, idx], y_bb[tr], sc.transform(X_busi.values)[:, idx],
                            g_bb[tr], n_boot=N_BOOT_TABPFN if A_NAME == 'TabPFN' else 1)
    LA.append(mu); SA.append(sd); print(f'  fold {f} model applied to BUSI')
EXT = {'Expert A (radiomics)': pd.DataFrame({'logit_mean': np.mean(LA, 0), 'logit_sd': np.mean(SA, 0) if A_NAME == 'TabPFN' else np.nan})}

## 5. Crops for BUSI (identical to Part 2) and the dataset / model classes of Experts B and C

In [ ]:
import torchvision.transforms.v2 as T2
from torchvision import tv_tensors
from torch.utils.data import Dataset, DataLoader
import torch.nn.utils.parametrize as P

def make_crop(img_path, mask_path, out=CACHE_SIZE, margin=0.25):
    img = np.array(Image.open(img_path).convert('L'), dtype=np.uint8)
    les = np.array(Image.open(mask_path).convert('L')) > 127
    if les.shape != img.shape:
        les = np.array(Image.fromarray(les.astype(np.uint8) * 255).resize(img.shape[::-1], Image.NEAREST)) > 127
    ring = ring_from(les); ys, xs = np.where(les | ring)
    cy, cx = (ys.min() + ys.max()) / 2, (xs.min() + xs.max()) / 2
    half = max(ys.max() - ys.min(), xs.max() - xs.min()) / 2 * (1 + margin) + 2
    y0, x0, s = int(round(cy - half)), int(round(cx - half)), int(round(2 * half))
    def cut(a):
        pad = np.zeros((s, s), a.dtype)
        sy0, sx0 = max(0, y0), max(0, x0); sy1, sx1 = min(a.shape[0], y0 + s), min(a.shape[1], x0 + s)
        pad[sy0 - y0:sy1 - y0, sx0 - x0:sx1 - x0] = a[sy0:sy1, sx0:sx1]; return pad
    im = np.array(Image.fromarray(cut(img)).resize((out, out), Image.BILINEAR))
    ml = np.array(Image.fromarray(cut(les.astype(np.uint8))).resize((out, out), Image.NEAREST))
    mr = np.array(Image.fromarray(cut(ring.astype(np.uint8))).resize((out, out), Image.NEAREST))
    return im, ml, mr

z = np.load(f'{P2_RES}/expertB_crops_{CACHE_SIZE}.npz', allow_pickle=True)
assert list(z['ids']) == list(master.id)
BB = dict(img=z['img'], les=z['les'], ring=z['ring'], y=y_bb)
cache = f'{RES_DIR}/busi_crops_{CACHE_SIZE}.npz'
if os.path.exists(cache) and list(np.load(cache, allow_pickle=True)['ids']) == list(busi.id):
    zz = np.load(cache, allow_pickle=True); BU = dict(img=zz['img'], les=zz['les'], ring=zz['ring'])
else:
    res = Parallel(n_jobs=-1)(delayed(make_crop)(r.img_path, r.mask_path) for r in busi.itertuples())
    BU = dict(zip(('img', 'les', 'ring'), (np.stack(a) for a in zip(*res))))
    np.savez_compressed(cache, **BU, ids=busi.id.values)
BU['y'] = busi.label.values
print('BUS-BRA crops', BB['img'].shape, '| BUSI crops', BU['img'].shape)

MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1); STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
GEOM = T2.Compose([T2.RandomHorizontalFlip(), T2.RandomAffine(degrees=10, translate=(0.05, 0.05), scale=(0.9, 1.1))])
PHOTO = T2.ColorJitter(brightness=0.2, contrast=0.2)

class CropDS(Dataset):                           # Part 2 (photo=True) and Part 3 (photo=False) inputs
    def __init__(self, data, idx, train, photo=True): self.d, self.idx, self.train, self.photo = data, np.asarray(idx), train, photo
    def __len__(self): return len(self.idx)
    def __getitem__(self, k):
        i = self.idx[k]
        im = tv_tensors.Image(torch.from_numpy(self.d['img'][i]).float().div(255)[None].repeat(3, 1, 1))
        ml = tv_tensors.Mask(torch.from_numpy(self.d['les'][i])[None]); mr = tv_tensors.Mask(torch.from_numpy(self.d['ring'][i])[None])
        if self.train:
            im, ml, mr = GEOM(im, ml, mr)
            if self.photo: im = PHOTO(im)
        im = F.interpolate(im[None], size=(IMG, IMG), mode='bilinear', align_corners=False)[0]
        ml = F.interpolate(ml[None].float(), size=(IMG, IMG), mode='nearest')[0]
        mr = F.interpolate(mr[None].float(), size=(IMG, IMG), mode='nearest')[0]
        return (im - MEAN) / STD, ml, mr, torch.tensor(float(self.d['y'][i]))

class DeepExpert(nn.Module):                     # Part 2
    def __init__(self, dual=True):
        super().__init__(); self.dual = dual
        self.backbone = timm.create_model(BACKBONE, pretrained=PRETRAINED, num_classes=0, drop_path_rate=0.1)
        d = self.backbone.num_features * (2 if dual else 1)
        self.head = nn.Sequential(nn.LayerNorm(d), nn.Dropout(0.3), nn.Linear(d, 1))
    def forward(self, x, ml, mr):
        e = torch.cat(self.backbone(torch.cat([x * ml, x * mr])).chunk(2), 1) if self.dual else self.backbone(x)
        return self.head(e).squeeze(1), e

## 6. Expert B: complete the saved models (identical training to Part 2) and apply them to BUSI
Models that already have weights are reused; the others are retrained with Part 2's code, split and seed, and saved. The retrained models' **BUS-BRA test-fold predictions** give a replica out-of-fold set (`oof_expertB_replica.csv`), used for the fusion so that BUS-BRA and BUSI inputs come from the same models. Re-run after a disconnection: finished models are skipped.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
def val_split(f):                                         # identical to Parts 2 and 3
    tr = np.where(master.fold != f)[0]
    a, b = next(StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=SEED + f).split(tr, y_bb[tr], g_bb[tr]))
    return tr[a], tr[b]

@torch.no_grad()
def predict_B(model, data, idx, tta=True):
    model.eval(); L = []
    for x, ml, mr, _ in DataLoader(CropDS(data, idx, False), batch_size=64, num_workers=2):
        x, ml, mr = x.to(device), ml.to(device), mr.to(device)
        with torch.autocast('cuda', dtype=torch.float16, enabled=device == 'cuda'):
            lo, _ = model(x, ml, mr)
            if tta: lo = (lo + model(x.flip(-1), ml.flip(-1), mr.flip(-1))[0]) / 2
        L.append(lo.float().cpu())
    return torch.cat(L).numpy()

def get_B(f, seed):
    out = f'{B_RUN}/dual_f{f}_s{seed}.npz'
    if os.path.exists(out): return dict(np.load(out))
    fit_idx, val_idx = val_split(f); test_idx = np.where(master.fold == f)[0]
    model = DeepExpert(True).to(device); w_old = f'{P2_RUN}/dual_f{f}_s{seed}.pt'
    if os.path.exists(w_old):                              # weights saved by Part 2
        model.load_state_dict(torch.load(w_old, map_location=device)); src = 'Part 2 weights'
        old = dict(np.load(f'{P2_RUN}/dual_f{f}_s{seed}.npz')); lv, lt = old['val_logit'], old['test_logit']
    else:                                                  # retrain exactly as Part 2
        torch.manual_seed(seed); np.random.seed(seed); src = 'retrained'
        dl = DataLoader(CropDS(BB, fit_idx, True), batch_size=BS_B, shuffle=True, num_workers=2, drop_last=True)
        opt = torch.optim.AdamW([{'params': list(model.backbone.parameters()), 'lr': LR_BACKBONE},
                                 {'params': list(model.head.parameters()), 'lr': LR_HEAD}], weight_decay=WD_B)
        steps, warm = EPOCHS_B * len(dl), len(dl)
        sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1, (s + 1) / warm) * 0.5 * (1 + np.cos(np.pi * min(1, s / steps))))
        pw = torch.tensor((1 - y_bb[fit_idx].mean()) / y_bb[fit_idx].mean(), device=device)
        scaler = torch.amp.GradScaler(enabled=device == 'cuda')
        for ep in range(EPOCHS_B):
            model.train(); t0 = time.time()
            for x, ml, mr, y in dl:
                x, ml, mr, y = x.to(device), ml.to(device), mr.to(device), y.to(device)
                with torch.autocast('cuda', dtype=torch.float16, enabled=device == 'cuda'):
                    lo, _ = model(x, ml, mr); loss = F.binary_cross_entropy_with_logits(lo.float(), y, pos_weight=pw)
                opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            print(f'    [B f{f} s{seed}] epoch {ep + 1}/{EPOCHS_B} ({time.time() - t0:.0f}s)')
        torch.save(model.state_dict(), f'{B_RUN}/dual_f{f}_s{seed}.pt')
        lv, lt = predict_B(model, BB, val_idx), predict_B(model, BB, test_idx)
    le = predict_B(model, BU, np.arange(len(busi)))
    res = dict(val_idx=val_idx, val_logit=lv, test_idx=test_idx, test_logit=lt, busi_logit=le)
    np.savez(out, **res); print(f'  B fold {f} seed {seed}: {src} | BUS-BRA test AUC {roc_auc_score(y_bb[test_idx], lt):.3f}')
    del model; torch.cuda.empty_cache(); return res

SEEDS_B = [SEED + s for s in range(N_SEEDS_B)]
T0 = time.time()
RB = {(f, s): get_B(f, s) for f in FOLDS for s in SEEDS_B}
print(f'Expert B models ready ({(time.time() - T0) / 60:.0f} min)')

B_rep = master[['id', 'patient', 'fold', 'label']].copy(); B_rep[['prob', 'logit_mean', 'logit_sd', 'thr']] = np.nan
LB, SB = [], []
for f in FOLDS:
    runs = [RB[(f, s)] for s in SEEDS_B]; te, va = runs[0]['test_idx'], runs[0]['val_idx']
    Lt = np.stack([r['test_logit'] for r in runs]); Lv = np.stack([r['val_logit'] for r in runs]).mean(0)
    Le = np.stack([r['busi_logit'] for r in runs])
    B_rep.loc[te, 'logit_mean'] = Lt.mean(0); B_rep.loc[te, 'prob'] = 1 / (1 + np.exp(-Lt.mean(0)))
    B_rep.loc[te, 'logit_sd'] = Lt.std(0) if len(runs) > 1 else np.nan
    B_rep.loc[te, 'thr'] = youden_threshold(y_bb[va], 1 / (1 + np.exp(-Lv)))
    LB.append(Le.mean(0)); SB.append(Le.std(0) if len(runs) > 1 else np.full(len(busi), np.nan))
save_oof(B_rep, 'expertB_replica', RES_DIR)
orig = pd.read_csv(f'{P2_RES}/oof_expertB.csv').set_index('id').loc[master.id]
print(f'Expert B BUS-BRA AUC: original Part 2 {roc_auc_score(y_bb, orig.prob):.3f} | replica used here {roc_auc_score(y_bb, B_rep.prob):.3f}')
EXT['Expert B (deep)'] = pd.DataFrame({'logit_mean': np.mean(LB, 0), 'logit_sd': np.mean(SB, 0)})

## 7. Experts C (RZ-USFM) and USFM-FT on BUSI: predictions, predicted concepts and the consistency score

In [ ]:
%%writefile {PART5_DIR}/usfm_colab.py
"""
usfm_colab.py — lightweight helpers to use the USFM foundation model
(Jiao et al., Medical Image Analysis 2024; https://github.com/openmedlab/USFM)
for classification / feature extraction WITHOUT installing mmcv / mmsegmentation.

Only needs: torch, timm, numpy, scipy (all available on Google Colab).
The ViT architecture is imported unchanged from the official repo
(usdsgen/modules/backbone/vision_transformer.py).
"""
import math
import os
import sys
from functools import partial

import numpy as np
import torch
import torch.nn as nn
from scipy.interpolate import RegularGridInterpolator as RGI

# USFM ViT-B configuration (configs/model/Cls/vit.yaml in the official repo)
USFM_VIT_B_CFG = dict(
    img_size=224,
    patch_size=16,
    in_chans=3,
    embed_dim=768,
    depth=12,
    num_heads=12,
    mlp_ratio=4,
    qkv_bias=True,
    attn_drop_rate=0.0,
    drop_path_rate=0.1,
    init_values=0.1,
    use_abs_pos_emb=False,
    use_rel_pos_bias=True,
    use_shared_rel_pos_bias=False,
    use_mean_pooling=True,
)

# USFM downstream preprocessing (usdsgen/data/datasets.py): 224x224, bicubic, ImageNet mean/std
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


def _import_vit(repo_dir):
    """Import the official USFM VisionTransformer class from the cloned repo."""
    repo_dir = os.path.abspath(repo_dir)
    if repo_dir not in sys.path:
        sys.path.insert(0, repo_dir)
    from usdsgen.modules.backbone.vision_transformer import VisionTransformer
    return VisionTransformer


def build_usfm(repo_dir="USFM", num_classes=0, **overrides):
    """Build the USFM ViT-B/16. num_classes=0 -> pure feature extractor (768-d output)."""
    VisionTransformer = _import_vit(repo_dir)
    cfg = {**USFM_VIT_B_CFG, **overrides}
    return VisionTransformer(
        num_classes=num_classes, norm_layer=partial(nn.LayerNorm, eps=1e-6), **cfg
    )


def _interp_rel_pos_table(table, src_size, dst_size, num_extra_tokens):
    """Geometric interpolation of a relative-position-bias table (as in BEiT / USFM)."""
    extra = table[-num_extra_tokens:, :]
    table = table[:-num_extra_tokens, :]

    def gp(a, r, n):
        return a * (1.0 - r**n) / (1.0 - r)

    left, right = 1.01, 1.5
    while right - left > 1e-6:
        q = (left + right) / 2.0
        if gp(1, q, src_size // 2) > dst_size // 2:
            right = q
        else:
            left = q
    dis, cur = [], 1
    for i in range(src_size // 2):
        dis.append(cur)
        cur += q ** (i + 1)
    r_ids = [-d for d in reversed(dis)]
    x = y = r_ids + [0] + dis
    t = dst_size // 2.0
    dx = dy = np.arange(-t, t + 0.1, 1.0)
    xi, yi = np.meshgrid(dx, dy, indexing="ij")
    pts = np.array([xi.ravel(), yi.ravel()]).T
    out = []
    for h in range(table.shape[1]):
        z = table[:, h].view(src_size, src_size).float().numpy()
        f = RGI((x, y), z.T, method="cubic", bounds_error=False)
        out.append(torch.tensor(f(pts).reshape(xi.shape), dtype=torch.float32).reshape(-1, 1))
    return torch.cat([torch.cat(out, dim=-1), extra.float()], dim=0)


def load_usfm_weights(model, ckpt_path, verbose=True):
    """
    Robust loader for USFM_latest.pth.
    Handles: wrapped checkpoints ({'model': ...}), 'module.'/'encoder.' prefixes,
    a shared relative-position bias (expanded to every block), and table-size
    interpolation. (The official remap function raises a KeyError when the shared
    table is absent; this version handles both layouts.)
    Returns a dict with loading statistics.
    """
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    for key in ("model", "state_dict", "module"):
        if isinstance(ckpt, dict) and key in ckpt and isinstance(ckpt[key], dict):
            ckpt = ckpt[key]

    sd = {}
    for k, v in ckpt.items():
        if not torch.is_tensor(v):
            continue
        for p in ("module.", "encoder.", "backbone.", "model."):
            if k.startswith(p):
                k = k[len(p):]
        sd[k] = v

    shared_key = "rel_pos_bias.relative_position_bias_table"
    if shared_key in sd:
        shared = sd.pop(shared_key)
        for i in range(len(model.blocks)):
            sd.setdefault(f"blocks.{i}.attn.relative_position_bias_table", shared.clone())
    sd = {k: v for k, v in sd.items() if "relative_position_index" not in k}

    msd = model.state_dict()
    gh, gw = model.patch_embed.patch_shape
    for k in list(sd.keys()):
        if "relative_position_bias_table" in k and k in msd and sd[k].shape != msd[k].shape:
            num_extra = msd[k].shape[0] - (2 * gh - 1) * (2 * gw - 1)
            src = int(round((sd[k].shape[0] - num_extra) ** 0.5))
            dst = int(round((msd[k].shape[0] - num_extra) ** 0.5))
            sd[k] = _interp_rel_pos_table(sd[k], src, dst, num_extra)

    skipped_shape = [k for k in sd if k in msd and sd[k].shape != msd[k].shape]
    for k in skipped_shape:
        sd.pop(k)
    skipped_head = [k for k in list(sd) if k.startswith("head.")]
    for k in skipped_head:  # never load a pretraining head into a new classifier
        sd.pop(k)

    missing, unexpected = model.load_state_dict(sd, strict=False)
    # relative_position_index buffers are rebuilt at init, so they are not "weights"
    backbone_keys = [k for k in msd if not k.startswith(("head.", "fc_norm."))
                     and "relative_position_index" not in k]
    loaded = [k for k in backbone_keys if k not in missing]
    stats = dict(
        backbone_params=len(backbone_keys),
        loaded=len(loaded),
        missing=[k for k in missing if not k.startswith(("head.", "fc_norm."))
                 and "relative_position_index" not in k],
        unexpected=list(unexpected),
        skipped_shape=skipped_shape,
    )
    if verbose:
        print(f"USFM backbone tensors loaded: {stats['loaded']}/{stats['backbone_params']}")
        if stats["missing"]:
            print("  missing (randomly initialised):", stats["missing"][:10], "...")
        if stats["unexpected"]:
            print("  unexpected (ignored):", stats["unexpected"][:10], "...")
        if skipped_shape:
            print("  skipped (shape mismatch):", skipped_shape[:10])
    if stats["loaded"] < 0.9 * stats["backbone_params"]:
        raise RuntimeError(
            "Less than 90% of USFM backbone weights were loaded - check the checkpoint "
            "file (is it the full USFM_latest.pth and not an HTML page from Google Drive?)."
        )
    return stats


In [ ]:
import gdown
from sklearn.covariance import LedoitWolf
if not os.path.exists(USFM_CKPT) or os.path.getsize(USFM_CKPT) < 50e6:
    os.makedirs(os.path.dirname(USFM_CKPT), exist_ok=True); gdown.download(id='1KRwXZgYterH895Z8EpXpR1L1eSMMJo4q', output=USFM_CKPT)
from usfm_colab import build_usfm, load_usfm_weights
_t = build_usfm(USFM_REPO, num_classes=0, img_size=IMG); load_usfm_weights(_t, USFM_CKPT)
BASE_SD = {k: v.clone() for k, v in _t.state_dict().items()}; del _t

class LoRA(nn.Module):
    def __init__(self, out_f, in_f, r=LORA_RANK, alpha=LORA_ALPHA):
        super().__init__(); self.A = nn.Parameter(torch.zeros(r, in_f)); self.B = nn.Parameter(torch.zeros(out_f, r)); self.scale = alpha / r
    def forward(self, W): return W + self.scale * self.B @ self.A
def mask_pool(t, m):
    a = F.avg_pool2d(m, 16).flatten(1); return (a.unsqueeze(-1) * t).sum(1) / (a.sum(1, keepdim=True) + 1e-6)
class RZUSFM(nn.Module):                         # Part 3
    def __init__(self, n_concepts=K, q=16):
        super().__init__()
        self.enc = build_usfm(USFM_REPO, num_classes=0, img_size=IMG); self.enc.load_state_dict(BASE_SD)
        for p in self.enc.parameters(): p.requires_grad = False
        for blk in self.enc.blocks: P.register_parametrization(blk.attn.qkv, 'weight', LoRA(*blk.attn.qkv.weight.shape))
        d = self.enc.embed_dim
        self.norms = nn.ModuleList([nn.LayerNorm(d) for _ in range(3)])
        self.rad_head = nn.Sequential(nn.Linear(3 * d, 256), nn.GELU(), nn.Dropout(0.1), nn.Linear(256, 2 * n_concepts))
        self.side = nn.Sequential(nn.Dropout(0.2), nn.Linear(3 * d, q))
        self.u = nn.Linear(n_concepts, 1, bias=False); self.v = nn.Linear(q, 1); self.K = n_concepts
    def forward(self, x, ml, mr):
        e_ = self.enc; t = e_.patch_embed(x); t = torch.cat([e_.cls_token.expand(t.size(0), -1, -1), t], 1)
        for blk in e_.blocks: t = blk(t)
        t = t[:, 1:]
        e = torch.cat([n(p) for n, p in zip(self.norms, [mask_pool(t, ml), mask_pool(t, mr), t.mean(1)])], 1)
        out = self.rad_head(e); r_hat = out[:, :self.K]
        return self.u(r_hat).squeeze(1) + self.v(self.side(e)).squeeze(1), r_hat

@torch.no_grad()
def predict_C(model, data, idx):
    model.eval(); L, R = [], []
    for x, ml, mr, _ in DataLoader(CropDS(data, idx, False, photo=False), batch_size=64, num_workers=2):
        x, ml, mr = x.to(device), ml.to(device), mr.to(device)
        with torch.autocast('cuda', dtype=torch.float16, enabled=device == 'cuda'):
            lo, rh = model(x, ml, mr); lo2, rh2 = model(x.flip(-1), ml.flip(-1), mr.flip(-1))
        L.append(((lo + lo2) / 2).float().cpu()); R.append(((rh + rh2) / 2).float().cpu())
    return torch.cat(L).numpy(), torch.cat(R).numpy()

sel = pd.read_csv(f'{P3_RES}/expertC_lambda_selection.csv'); pos = sel[sel['lambda'] > 0]
LAM_STAR = pos.loc[pos.groupby('fold').val_AUC.idxmax()].set_index('fold')['lambda'].to_dict()
SEEDS_C = [SEED + s for s in range(N_SEEDS_C)]
print('λ* per fold (from Part 3):', LAM_STAR)
tag = lambda lam, f, s: f'lam{lam:g}_f{f}_s{s}'

def apply_variant(lam_of_fold, name):
    cache_f = f'{RES_DIR}/busi_{name}.npz'
    if os.path.exists(cache_f): return dict(np.load(cache_f))
    L_all, S_all, R_all, C_all = [], [], [], []
    for f in FOLDS:
        Ls, Rs, Rv = [], [], []
        for s in SEEDS_C:
            t = tag(lam_of_fold[f], f, s)
            m = RZUSFM().to(device); missing = m.load_state_dict(torch.load(f'{P3_RUN}/{t}.pt', map_location=device), strict=False)
            assert not [k for k in missing.missing_keys if 'parametrizations.weight.0' in k or not k.startswith('enc.')], missing.missing_keys[:5]
            lo, rh = predict_C(m, BU, np.arange(len(busi))); Ls.append(lo); Rs.append(rh)
            Rv.append(np.load(f'{P3_RUN}/{t}.npz')['val_rhat']); del m; torch.cuda.empty_cache()
        va = np.load(f'{P3_RUN}/{tag(lam_of_fold[f], f, SEEDS_C[0])}.npz')['val_idx']
        Rh = np.mean(Rs, 0); lw = LedoitWolf().fit(CONC_BB[f][va] - np.mean(Rv, 0))
        d = CONC_BUSI[f] - Rh; C_all.append(np.einsum('ij,jk,ik->i', d, lw.precision_, d))
        L_all.append(np.mean(Ls, 0)); S_all.append(np.std(Ls, 0) if len(Ls) > 1 else np.full(len(busi), np.nan)); R_all.append(Rh)
        print(f'  {name}: fold {f} done')
    out = dict(logit=np.mean(L_all, 0), sd=np.mean(S_all, 0), consistency=np.mean(C_all, 0), rhat=np.stack(R_all))
    np.savez(cache_f, **out); return out

C_ext = apply_variant(LAM_STAR, 'RZ-USFM')
FT_ext = apply_variant({f: 0.0 for f in FOLDS}, 'USFM-FT')
EXT['Expert C (RZ-USFM)'] = pd.DataFrame({'logit_mean': C_ext['logit'], 'logit_sd': C_ext['sd'], 'consistency': C_ext['consistency']})
EXT['USFM-FT (λ=0)'] = pd.DataFrame({'logit_mean': FT_ext['logit'], 'logit_sd': FT_ext['sd'], 'consistency': FT_ext['consistency']})

r2 = lambda r, rh: 1 - ((r - rh) ** 2).sum(0) / (((r - r.mean(0)) ** 2).sum(0) + 1e-8)
r2_ext = pd.DataFrame([r2(CONC_BUSI[f], C_ext['rhat'][j]) for j, f in enumerate(FOLDS)], columns=CONCEPT_NAMES).mean()
int_tab = pd.read_csv(f'{P3_RES}/expertC_concept_R2.csv', index_col=0)
r2_cmp = pd.DataFrame({'BUS-BRA (internal)': int_tab['RZ-USFM head'], 'BUSI (external)': r2_ext}); display(r2_cmp.round(3))
r2_cmp.to_csv(f'{RES_DIR}/concept_R2_internal_vs_external.csv')
print('mean concept R² of the RZ-USFM radiomics head:', r2_cmp.mean().round(3).to_dict())

## 8. Assemble internal (BUS-BRA, out-of-fold) and external (BUSI) predictions of the four single models
Decision thresholds are fixed on BUS-BRA (Youden on the out-of-fold predictions) — BUSI is never used to choose anything.

In [ ]:
def load_oof(path, extra=None):
    o = pd.read_csv(path).set_index('id').loc[master.id].reset_index()
    if extra: o = o.merge(pd.read_csv(extra)[['id', 'consistency']], on='id', how='left')
    return o
INT = {'Expert A (radiomics)': load_oof(f'{P1_RES}/oof_expertA.csv'),
       'Expert B (deep)': B_rep.copy(),
       'Expert C (RZ-USFM)': load_oof(f'{P3_RES}/oof_expertC.csv', f'{P3_RES}/expertC_extra.csv'),
       'USFM-FT (λ=0)': load_oof(f'{P3_RES}/oof_USFM-FT.csv', f'{P3_RES}/USFM-FT_extra.csv')}

def ext_frame(lm, thr, sd=None, extra=None):
    o = busi[['id', 'patient', 'fold', 'label']].copy(); o['logit_mean'] = lm; o['prob'] = 1 / (1 + np.exp(-lm))
    o['logit_sd'] = np.nan if sd is None else sd; o['thr'] = thr
    if extra is not None: o['consistency'] = extra
    return o
for n in list(EXT):
    thr = youden_threshold(y_bb, INT[n].prob.values)
    EXT[n] = ext_frame(EXT[n].logit_mean.values, thr, EXT[n].logit_sd.values, EXT[n]['consistency'].values if 'consistency' in EXT[n] else None)
    print(f'{n:22s} AUC  BUS-BRA {roc_auc_score(y_bb, INT[n].prob):.3f}  ->  BUSI {roc_auc_score(busi.label, EXT[n].prob):.3f}')

## 9. Fusion models refitted on all BUS-BRA out-of-fold predictions and applied to BUSI (unseen scanner)

In [ ]:
import pymc as pm, pytensor.tensor as pt

def meta(experts):
    mu = np.column_stack([o.logit_mean.values for o in experts]); u = np.column_stack([o.logit_sd.fillna(0).values for o in experts])
    has_u = np.array([o.logit_sd.notna().all() for o in experts], float)
    c = np.log1p(experts[-1]['consistency'].values) if 'consistency' in experts[-1] else np.zeros(len(mu))
    return dict(mu=mu, u=u, has_u=has_u, c=c)

def standardise_with(Mfit, M):                   # statistics from BUS-BRA only
    lu_f = np.log(Mfit['u'] + 1e-3); m_, s_ = lu_f.mean(0), lu_f.std(0) + 1e-8
    cm, cs = Mfit['c'].mean(), Mfit['c'].std() + 1e-8
    return (np.log(M['u'] + 1e-3) - m_) / s_ * Mfit['has_u'], (M['c'] - cm) / cs

def PV(idata, var):
    v = np.asarray(idata.posterior[var].values); return v.reshape(-1, *v.shape[2:])

def fit_bayes(M, gated, scanner, use_c):
    lu, cc = standardise_with(M, M); K_ = M['mu'].shape[1]; S = len(DEV_LEVELS) if scanner else 1
    s = S_IDX if scanner else np.zeros(len(S_IDX), int); prev = y_bb.mean()
    with pm.Model():
        b0 = pm.Normal('b0', np.log(prev / (1 - prev)), 1); abar = pm.Normal('abar', 0, 1, shape=K_)
        if scanner and S > 1:
            tau = pm.HalfNormal('tau', 0.5, shape=K_); zz = pm.Normal('z', 0, 1, shape=(S, K_)); alpha = abar + tau * zz
        else:
            alpha = pt.tile(abar[None, :], (S, 1))
        lin = alpha[s]
        if gated:
            lin = lin + pm.Normal('gamma', 0, 0.5, shape=K_) * M['has_u'] * lu
            if use_c: lin = lin + pm.Normal('delta', 0, 0.5, shape=K_) * cc[:, None]
        w = pt.softplus(lin); eta = b0 + (w * M['mu']).sum(1); v = (w ** 2 * M['u'] ** 2).sum(1)
        pm.Bernoulli('y', p=pm.math.sigmoid(eta / pt.sqrt(1 + np.pi * v / 8)), observed=y_bb)
        idata = pm.sample(DRAWS, tune=TUNE, chains=CHAINS, cores=CHAINS, target_accept=0.9, progressbar=False, random_seed=SEED)
    return dict(idata=idata, gated=gated, scanner=scanner and S > 1, use_c=use_c, Mfit=M)

def predict_bayes(Fm, M, new_scanner, n_draws=600, return_w=False):
    I = Fm['idata']; b0a = PV(I, 'b0'); sel_ = np.linspace(0, len(b0a) - 1, n_draws).astype(int)
    b0, abar = b0a[sel_], PV(I, 'abar')[sel_]; n = len(M['mu']); rng = np.random.default_rng(SEED)
    if Fm['scanner'] and new_scanner:            # unseen scanner: alpha = abar + tau * z, z ~ N(0, 1) shared by all lesions
        A = (abar + PV(I, 'tau')[sel_] * rng.standard_normal(abar.shape))[:, None, :].repeat(n, 1)
    elif Fm['scanner']:
        A = (abar[:, None, :] + PV(I, 'tau')[sel_][:, None, :] * PV(I, 'z')[sel_])[:, S_IDX, :]
    else:
        A = abar[:, None, :].repeat(n, 1)
    lu, cc = standardise_with(Fm['Mfit'], M); lin = A.copy()
    if Fm['gated']:
        lin += PV(I, 'gamma')[sel_][:, None, :] * (M['has_u'] * lu)[None]
        if Fm['use_c']: lin += PV(I, 'delta')[sel_][:, None, :] * cc[None, :, None]
    w = np.logaddexp(0, lin); eta = b0[:, None] + (w * M['mu'][None]).sum(-1); v = (w ** 2 * M['u'][None] ** 2).sum(-1)
    p = 1 / (1 + np.exp(-eta / np.sqrt(1 + np.pi * v / 8)))
    return (p.mean(0), w.mean(0)) if return_w else p.mean(0)

A_i, B_i, C_i, FT_i = (INT[k] for k in ('Expert A (radiomics)', 'Expert B (deep)', 'Expert C (RZ-USFM)', 'USFM-FT (λ=0)'))
A_e, B_e, C_e, FT_e = (EXT[k] for k in ('Expert A (radiomics)', 'Expert B (deep)', 'Expert C (RZ-USFM)', 'USFM-FT (λ=0)'))
Mi_RZ, Me_RZ, Mi_FT, Me_FT = meta([A_i, B_i, C_i]), meta([A_e, B_e, C_e]), meta([A_i, B_i, FT_i]), meta([A_e, B_e, FT_e])
Me_RZ['has_u'] = Mi_RZ['has_u'].copy(); Me_FT['has_u'] = Mi_FT['has_u'].copy()

FUS_INT, FUS_EXT, WEIGHTS = {}, {}, {}
def add(name, p_int, p_ext):
    thr = youden_threshold(y_bb, p_int)
    o = master[['id', 'patient', 'fold', 'label']].copy(); o['prob'] = p_int; o['logit_mean'] = logit(p_int); o['logit_sd'] = np.nan; o['thr'] = thr
    FUS_INT[name] = o; FUS_EXT[name] = ext_frame(logit(p_ext), thr)
add('Average (A+B+C)', np.mean([o.prob for o in (A_i, B_i, C_i)], 0), np.mean([o.prob for o in (A_e, B_e, C_e)], 0))
lr = LogisticRegression(C=1.0, max_iter=5000).fit(Mi_RZ['mu'], y_bb)
add('LR stacking', lr.predict_proba(Mi_RZ['mu'])[:, 1], lr.predict_proba(Me_RZ['mu'])[:, 1])
BAYES = {'Bayesian stacking (ungated)': (Mi_RZ, Me_RZ, False, False, False),
         'Proposed (gated Bayesian fusion)': (Mi_RZ, Me_RZ, True, True, True),
         'Ablation - no consistency score': (Mi_RZ, Me_RZ, True, True, False),
         'Ablation - USFM-FT instead of RZ-USFM': (Mi_FT, Me_FT, True, True, True)}
FITS = {}
for name, (Mi, Me, gated, scanner, use_c) in BAYES.items():
    t = time.time(); Fm = fit_bayes(Mi, gated, scanner, use_c); FITS[name] = Fm
    p_int, w_int = predict_bayes(Fm, Mi, False, return_w=True); p_ext, w_ext = predict_bayes(Fm, Me, True, return_w=True)
    add(name, p_int, p_ext); WEIGHTS[name] = (w_int, w_ext)
    print(f'{name:40s} BUSI AUC {roc_auc_score(busi.label, p_ext):.3f} ({time.time() - t:.0f}s)')
for n, o in FUS_EXT.items(): save_oof(o, f'busi_{re.sub(r"[^A-Za-z0-9]+", "_", n)}', RES_DIR)

## 10. External results: metrics (95% CI), internal → external change, DeLong tests on BUSI

In [ ]:
ALL_EXT = {**EXT, **FUS_EXT}
rows_ext = [evaluate_oof(o, n, RES_DIR) for n, o in ALL_EXT.items()]
METRICS = ('AUC', 'Accuracy', 'Balanced_Acc', 'Sensitivity', 'Specificity', 'Precision', 'F1', 'Brier')
tab_ext = summary_table(rows_ext, metrics=METRICS); display(tab_ext); tab_ext.to_csv(f'{RES_DIR}/external_BUSI_table.csv')
pd.DataFrame(rows_ext).to_csv(f'{RES_DIR}/external_BUSI_full.csv', index=False)

p4 = pd.read_csv(f'{P4_RES}/final_comparison_full.csv').set_index('model') if P4_RES else None
gap = []
for r in rows_ext:
    n = r['model']
    if n in INT: ri = evaluate_oof(INT[n], f'internal_{n}', RES_DIR, n_boot=500); ai, ci_i = ri['AUC'], ri['AUC_CI']
    elif p4 is not None and n in p4.index: ai, ci_i = p4.loc[n, 'AUC'], p4.loc[n, 'AUC_CI']
    else: ai, ci_i = roc_auc_score(y_bb, FUS_INT[n].prob), ''
    gap.append({'model': n, 'AUC BUS-BRA (internal)': ai, 'CI int': ci_i, 'AUC BUSI (external)': r['AUC'], 'CI ext': r['AUC_CI'],
                'ΔAUC (ext − int)': r['AUC'] - ai, 'Brier BUSI': r['Brier']})
gap = pd.DataFrame(gap); display(gap.round(3)); gap.to_csv(f'{RES_DIR}/generalisation_gap.csv', index=False)

def delong(y, p1, p2):
    y = np.asarray(y).astype(bool); Pm = np.vstack([p1, p2]); m, n = y.sum(), (~y).sum()
    def midrank(x):
        o = np.argsort(x); xs = x[o]; r = np.zeros(len(x)); i = 0
        while i < len(x):
            j = i
            while j < len(x) and xs[j] == xs[i]: j += 1
            r[i:j] = 0.5 * (i + j - 1) + 1; i = j
        out = np.empty(len(x)); out[o] = r; return out
    tx = np.array([midrank(a) for a in Pm[:, y]]); ty = np.array([midrank(a) for a in Pm[:, ~y]]); tz = np.array([midrank(a) for a in Pm])
    auc = tz[:, y].sum(1) / m / n - (m + 1) / (2 * n); v01 = (tz[:, y] - tx) / n; v10 = 1 - (tz[:, ~y] - ty) / m
    S = np.cov(v01) / m + np.cov(v10) / n
    return auc[0], auc[1], 2 * stats.norm.sf(abs((auc[0] - auc[1]) / np.sqrt(S[0, 0] + S[1, 1] - 2 * S[0, 1] + 1e-12)))
yb = busi.label.values
PAIRS = [('Expert C (RZ-USFM)', 'USFM-FT (λ=0)'), ('Expert C (RZ-USFM)', 'Expert B (deep)'), ('Expert C (RZ-USFM)', 'Expert A (radiomics)'),
         ('Proposed (gated Bayesian fusion)', 'Bayesian stacking (ungated)'), ('Bayesian stacking (ungated)', 'Expert B (deep)'),
         ('Bayesian stacking (ungated)', 'LR stacking'), ('Proposed (gated Bayesian fusion)', 'Ablation - USFM-FT instead of RZ-USFM')]
dl_rows = [{'model 1': a, 'model 2': b, **dict(zip(('AUC 1', 'AUC 2', 'p (DeLong)'), delong(yb, ALL_EXT[a].prob.values, ALL_EXT[b].prob.values)))}
           for a, b in PAIRS]
dl_df = pd.DataFrame(dl_rows); dl_df['ΔAUC'] = dl_df['AUC 1'] - dl_df['AUC 2']; display(dl_df.round(4)); dl_df.to_csv(f'{RES_DIR}/delong_BUSI.csv', index=False)

## 11. Figures: ROC, calibration, confusion matrices, metric bars and the generalisation gap

In [ ]:
MAIN = ['Expert A (radiomics)', 'Expert B (deep)', 'Expert C (RZ-USFM)', 'USFM-FT (λ=0)', 'LR stacking',
        'Bayesian stacking (ungated)', 'Proposed (gated Bayesian fusion)']
plot_roc({n: ALL_EXT[n] for n in MAIN}, f'{FIG_DIR}/roc_BUSI.png', 'External validation on BUSI (trained on BUS-BRA)')
plot_pr({n: ALL_EXT[n] for n in MAIN}, f'{FIG_DIR}/pr_BUSI.png')
plot_calibration({n: ALL_EXT[n] for n in MAIN}, f'{FIG_DIR}/calibration_BUSI.png')
safe = lambda n: re.sub(r'[^A-Za-z0-9_-]+', '_', n).strip('_')
for n in ('Expert B (deep)', 'Expert C (RZ-USFM)', 'Bayesian stacking (ungated)', 'Proposed (gated Bayesian fusion)'):
    plot_confusion(ALL_EXT[n], n + ' — BUSI', f'{FIG_DIR}/cm_BUSI_{safe(n)}.png')
plot_metric_bars([r for r in rows_ext if r['model'] in MAIN], f'{FIG_DIR}/metrics_bars_BUSI.png')

g = gap.set_index('model').loc[[m for m in MAIN if m in gap.model.values]]
fig, ax = plt.subplots(figsize=(7.5, 0.55 * len(g) + 1.2)); yy = np.arange(len(g))[::-1]
for k, (n, r) in enumerate(g.iterrows()):
    ax.plot([r['AUC BUSI (external)'], r['AUC BUS-BRA (internal)']], [yy[k]] * 2, '-', c='grey', lw=1.5)
    for col, key, mk in (('#4C72B0', 'AUC BUS-BRA (internal)', 'CI int'), ('#C44E52', 'AUC BUSI (external)', 'CI ext')):
        lo_hi = [float(v) for v in str(r[mk]).split('-')] if r[mk] else [r[key]] * 2
        ax.errorbar(r[key], yy[k], xerr=[[r[key] - lo_hi[0]], [lo_hi[1] - r[key]]], fmt='o', c=col, capsize=3)
ax.set_yticks(yy, g.index, fontsize=8); ax.set_xlabel('AUC (95% CI)')
ax.plot([], [], 'o', c='#4C72B0', label='BUS-BRA (internal, out-of-fold)'); ax.plot([], [], 'o', c='#C44E52', label='BUSI (external)')
ax.legend(fontsize=8, loc='lower left'); ax.set_title('Generalisation gap: BUS-BRA → BUSI')
fig.savefig(f'{FIG_DIR}/generalisation_gap.png', bbox_inches='tight'); plt.show()

## 12. Reliability under distribution shift
Does the radiomic consistency score $c$ increase on BUSI, and do $c$ and the expert uncertainties flag BUSI errors? How do the fusion weights move?

In [ ]:
c_int, c_ext = np.log1p(INT['Expert C (RZ-USFM)'].consistency.values), np.log1p(EXT['Expert C (RZ-USFM)'].consistency.values)
print(f'log(1+c): BUS-BRA median {np.median(c_int):.2f} | BUSI median {np.median(c_ext):.2f} | '
      f'Mann-Whitney p = {stats.mannwhitneyu(c_int, c_ext).pvalue:.2e} | AUROC (BUSI vs BUS-BRA) = {roc_auc_score(np.r_[np.zeros(len(c_int)), np.ones(len(c_ext))], np.r_[c_int, c_ext]):.3f}')
rel = []
for n in ('Expert A (radiomics)', 'Expert B (deep)', 'Expert C (RZ-USFM)', 'USFM-FT (λ=0)'):
    for split, o in (('BUS-BRA', INT[n]), ('BUSI', EXT[n])):
        err = ((o.prob >= o.thr).astype(int) != o.label).astype(int).values
        row = {'model': n, 'data': split, 'error rate': err.mean()}
        if o.logit_sd.notna().all() and len(np.unique(err)) == 2: row['AUROC(uncertainty → error)'] = roc_auc_score(err, o.logit_sd)
        if 'consistency' in o and len(np.unique(err)) == 2: row['AUROC(c → error)'] = roc_auc_score(err, np.log1p(o.consistency))
        rel.append(row)
rel = pd.DataFrame(rel); display(rel.round(3)); rel.to_csv(f'{RES_DIR}/reliability_under_shift.csv', index=False)

EXPN = ['A: radiomics', 'B: deep', 'C: RZ-USFM']
w_int, w_ext = WEIGHTS['Proposed (gated Bayesian fusion)']
s_int, s_ext = w_int / w_int.sum(1, keepdims=True), w_ext / w_ext.sum(1, keepdims=True)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].hist(c_int, bins=40, alpha=.6, density=True, label='BUS-BRA'); axes[0].hist(c_ext, bins=40, alpha=.6, density=True, label='BUSI')
axes[0].set(xlabel='log(1 + c)', ylabel='Density', title='Radiomic consistency score under shift'); axes[0].legend()
pos_ = np.arange(len(EXPN))
axes[1].violinplot([s_int[:, k] for k in range(3)], positions=pos_ - .18, widths=.3, showmeans=True)
axes[1].violinplot([s_ext[:, k] for k in range(3)], positions=pos_ + .18, widths=.3, showmeans=True)
axes[1].set_xticks(pos_, EXPN); axes[1].set(ylabel='Share of fusion weight', title='Gated fusion weights: BUS-BRA (left) vs BUSI (right)')
fig.tight_layout(); fig.savefig(f'{FIG_DIR}/reliability_under_shift.png', bbox_inches='tight'); plt.show()
print('mean weight share BUS-BRA:', dict(zip(EXPN, s_int.mean(0).round(3))), '| BUSI:', dict(zip(EXPN, s_ext.mean(0).round(3))))

## 13. Conformal triage with bootstrap CIs and McNemar tests
- **Calibration on BUS-BRA only** (out-of-fold scores: Part 4's cross-validated fusion predictions; the experts' OOF predictions).
- **BUS-BRA BI-RADS 4:** cross-fitted as in Part 4 (calibrate on the other folds), now with patient-level bootstrap 95% CIs.
- **BUSI:** cut-offs from all BUS-BRA scores applied unchanged → tests whether the sensitivity guarantee transfers under shift.
- **McNemar** (exact): which benign lesions each pair of models sends home.

In [ ]:
def cutoffs(cal_p, cal_y, a1=ALPHA_MISS, a0=ALPHA_FP):
    s1 = np.sort(cal_p[cal_y == 1]); k1 = int(np.floor(a1 * (len(s1) + 1)))
    s0 = np.sort(cal_p[cal_y == 0])[::-1]; k0 = int(np.floor(a0 * (len(s0) + 1)))
    return (s1[k1 - 1] if k1 >= 1 else -np.inf), (s0[k0 - 1] if k0 >= 1 else np.inf)
def decide(p, t_low, t_high): return np.where(p < t_low, 'avoid', np.where(p > t_high, 'malignant', 'uncertain'))

cache4 = dict(np.load(f'{P4_RES}/fusion_oof_cache.npz', allow_pickle=True)) if P4_RES else {}
def oof_scores(name):                          # genuinely out-of-fold BUS-BRA scores for calibration
    if name in INT: return INT[name].prob.values
    if all(f'{name}|prob|{f}' in cache4 for f in FOLDS):
        p = np.full(len(master), np.nan)
        for f in FOLDS: p[(master.fold == f).values] = cache4[f'{name}|prob|{f}']
        return p
    print(f'  note: no out-of-fold scores for {name}; using in-sample BUS-BRA fits (optimistic)'); return FUS_INT[name].prob.values

def summary(dec, y, groups, n_boot=2000):
    def stat(idx):
        d, yy = dec[idx], y[idx]; home = d == 'avoid'
        return 1 - (home & (yy == 1)).sum() / max(1, (yy == 1).sum()), (home & (yy == 0)).sum() / max(1, (yy == 0).sum())
    sens, av = stat(np.arange(len(y))); rng = np.random.default_rng(SEED); ug = np.unique(groups)
    ix = {g_: np.where(groups == g_)[0] for g_ in ug}; B_ = np.array([stat(np.concatenate([ix[g_] for g_ in rng.choice(ug, len(ug))])) for _ in range(n_boot)])
    return {'sensitivity': sens, 'sens 95% CI': '{:.3f}-{:.3f}'.format(*np.percentile(B_[:, 0], [2.5, 97.5])),
            'benign biopsies avoided': av, 'avoided 95% CI': '{:.3f}-{:.3f}'.format(*np.percentile(B_[:, 1], [2.5, 97.5])),
            'n benign avoided': int(((dec == 'avoid') & (y == 0)).sum()), 'n cancers sent home': int(((dec == 'avoid') & (y == 1)).sum()),
            'uncertain rate': (dec == 'uncertain').mean()}

TRI = ['Expert A (radiomics)', 'Expert B (deep)', 'Expert C (RZ-USFM)', 'LR stacking', 'Bayesian stacking (ungated)',
       'Proposed (gated Bayesian fusion)', 'Ablation - USFM-FT instead of RZ-USFM']
DEC = {}; tri = []
for n in TRI:
    cal = oof_scores(n)
    d_int = np.empty(len(master), object)
    for f in FOLDS:
        te = (master.fold == f).values; tl, th = cutoffs(cal[~te], y_bb[~te]); d_int[te] = decide(cal[te], tl, th)
    tl, th = cutoffs(cal, y_bb); d_ext = decide(ALL_EXT[n].prob.values, tl, th)
    DEC[n] = (d_int, d_ext)
    if BR4.sum(): tri.append({'model': n, 'data': 'BUS-BRA BI-RADS 4', **summary(d_int[BR4], y_bb[BR4], g_bb[BR4])})
    tri.append({'model': n, 'data': 'BUSI (external)', **summary(d_ext, yb, busi.id.values)})
tri = pd.DataFrame(tri); display(tri.round(3)); tri.to_csv(f'{RES_DIR}/conformal_triage_with_CI.csv', index=False)

def mcnemar(d1, d2, y, mask):
    a = (d1 == 'avoid')[mask & (y == 0)]; b = (d2 == 'avoid')[mask & (y == 0)]
    n01, n10 = int((a & ~b).sum()), int((~a & b).sum())
    return n01, n10, (stats.binomtest(n01, n01 + n10, 0.5).pvalue if n01 + n10 else 1.0)
mc = []
for a, b in [('Bayesian stacking (ungated)', 'Proposed (gated Bayesian fusion)'), ('Bayesian stacking (ungated)', 'Expert B (deep)'),
             ('Bayesian stacking (ungated)', 'LR stacking'), ('Expert C (RZ-USFM)', 'Expert B (deep)')]:
    if BR4.sum():
        n01, n10, p = mcnemar(DEC[a][0], DEC[b][0], y_bb, BR4)
        mc.append({'model 1': a, 'model 2': b, 'data': 'BUS-BRA BI-RADS 4', 'benign avoided only by 1': n01, 'only by 2': n10, 'p (exact McNemar)': p})
    n01, n10, p = mcnemar(DEC[a][1], DEC[b][1], yb, np.ones(len(yb), bool))
    mc.append({'model 1': a, 'model 2': b, 'data': 'BUSI', 'benign avoided only by 1': n01, 'only by 2': n10, 'p (exact McNemar)': p})
mc = pd.DataFrame(mc); display(mc.round(4)); mc.to_csv(f'{RES_DIR}/mcnemar_triage.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)
for ax, data in zip(axes, ['BUS-BRA BI-RADS 4', 'BUSI (external)']):
    t_ = tri[tri.data == data].set_index('model').reindex(TRI)
    if t_.empty or t_['benign biopsies avoided'].isna().all(): ax.set_visible(False); continue
    lo = t_['benign biopsies avoided'] - t_['avoided 95% CI'].str.split('-').str[0].astype(float)
    hi = t_['avoided 95% CI'].str.split('-').str[1].astype(float) - t_['benign biopsies avoided']
    ax.barh(range(len(t_)), 100 * t_['benign biopsies avoided'], xerr=[100 * lo, 100 * hi], color='#55A868', capsize=3)
    for k, (n, r) in enumerate(t_.iterrows()): ax.text(1, k, f"sens {100 * r['sensitivity']:.1f}%", va='center', fontsize=7, color='white')
    ax.set_yticks(range(len(t_)), t_.index, fontsize=8); ax.set(xlabel='Benign biopsies avoided (%) at α = 2%', title=data)
fig.tight_layout(); fig.savefig(f'{FIG_DIR}/triage_with_CI.png', bbox_inches='tight'); plt.show()

## 14. RZ-USFM on BUSI: concept-level explanations
Least-consistent BUSI lesions (highest $c$): image, measured vs RZ-USFM-predicted radiomic concepts; and global concept importance (exact SHAP of the linear bottleneck) on BUS-BRA vs BUSI.

In [ ]:
top = np.argsort(-EXT['Expert C (RZ-USFM)'].consistency.values)[:4]
Rh = C_ext['rhat'].mean(0); Rm = np.mean([CONC_BUSI[f] for f in FOLDS], 0)
fig, axes = plt.subplots(2, len(top), figsize=(4 * len(top), 7.5))
for j, i in enumerate(top):
    ov = np.stack([BU['img'][i]] * 3, -1).astype(float) / 255
    axes[0, j].imshow(ov); axes[0, j].contour(BU['les'][i], [0.5], colors='r', linewidths=1); axes[0, j].contour(BU['ring'][i], [0.5], colors='c', linewidths=.7)
    o = EXT['Expert C (RZ-USFM)']
    axes[0, j].set_title(f"{busi.id[i]} ({['benign', 'malignant'][yb[i]]})\np = {o.prob[i]:.2f}, c = {o.consistency[i]:.0f}", fontsize=8); axes[0, j].axis('off')
    axes[1, j].barh(np.arange(K) + .2, Rm[i], .4, label='measured'); axes[1, j].barh(np.arange(K) - .2, Rh[i], .4, label='RZ-USFM')
    axes[1, j].set_yticks(np.arange(K), CONCEPT_NAMES if j == 0 else [''] * K, fontsize=7)
axes[1, 0].legend(fontsize=7); fig.suptitle('Least radiomically consistent BUSI lesions'); fig.tight_layout()
fig.savefig(f'{FIG_DIR}/busi_low_consistency_cases.png', bbox_inches='tight'); plt.show()

U = np.stack([np.load(f'{P3_RUN}/{tag(LAM_STAR[f], f, SEED)}.npz')['u'] for f in FOLDS])
shap_ext = np.mean([np.abs(U[j] * (C_ext['rhat'][j] - C_ext['rhat'][j].mean(0))).mean(0) for j in range(len(FOLDS))], 0)
cs = pd.read_csv(f'{P3_RES}/expertC_concept_shap.csv').set_index('concept').reindex(CONCEPT_NAMES)
imp = pd.DataFrame({'BUS-BRA': cs.mean_abs_shap.values, 'BUSI': shap_ext}, index=CONCEPT_NAMES).sort_values('BUS-BRA')
ax = imp.plot.barh(figsize=(7, 0.4 * K + 1.5)); ax.set(xlabel='mean |SHAP| (exact, concept bottleneck)', title='Which radiomic concepts drive RZ-USFM: BUS-BRA vs BUSI')
plt.savefig(f'{FIG_DIR}/concept_importance_internal_vs_external.png', bbox_inches='tight'); plt.show()
imp.to_csv(f'{RES_DIR}/concept_importance_internal_vs_external.csv')
print('rank correlation of concept importance (BUS-BRA vs BUSI):', round(stats.spearmanr(imp['BUS-BRA'], imp['BUSI']).correlation, 3))

## 15. Outputs of Part 5 (all in `rsusfm_project_part5/`)
| File | Content |
|---|---|
| `results/busi_master.csv` | curated BUSI (after deduplication) |
| `results/external_BUSI_table.csv`, `external_BUSI_full.csv` | all models on BUSI, all metrics with 95% CI |
| `results/generalisation_gap.csv` | AUC BUS-BRA vs BUSI per model |
| `results/delong_BUSI.csv` | key pairwise AUC tests on BUSI (RZ-USFM vs USFM-FT, gated vs ungated, …) |
| `results/reliability_under_shift.csv`, `concept_R2_internal_vs_external.csv` | consistency score and uncertainties under shift; concept prediction on BUSI |
| `results/conformal_triage_with_CI.csv`, `mcnemar_triage.csv` | triage on BUS-BRA BI-RADS 4 and BUSI with bootstrap CIs and McNemar tests |
| `results/oof_expertB_replica.csv`, `expertB_models/` | completed Expert B ensemble (weights saved) |
| `figures/*.png` | ROC / PR / calibration / confusion matrices on BUSI, generalisation gap, reliability, triage, concept explanations |